# Fraud Detection Robustness Benchmark: Kaggle Dual T4 Research Run

This notebook runs the graph fraud-detection robustness benchmark on a Kaggle notebook with the GPU accelerator set to two NVIDIA T4 GPUs. It fetches the repository from `https://github.com/EndritHasani01/fraud-detection-robustness-benchmark`, validates the runtime, runs a fast smoke test, then runs the report-facing v3 benchmark stages.

The repository contract used here is the current v3 workflow: YelpChi from `dgl.data.FraudDataset`, deterministic graph variants, integrated `mlp`, `sage`, `pmp`, and `secgfd` models, both `train_on_variant` and `train_clean_eval_all` protocols, one unified `results.csv`, `variant_audit.csv`, and report-ready plot artifacts.

## Kaggle Runtime Requirements

Before running this notebook in Kaggle:

- Set `Settings > Accelerator` to `GPU T4 x2`.
- Turn `Internet` on so the notebook can clone GitHub and install DGL/PyTorch wheels.
- Keep outputs under `/kaggle/working`; Kaggle includes that directory when you save notebook output.

Kaggle uses Linux commands, so the benchmark is launched with `python -m benchmark.run`, not Windows `py -m benchmark.run`.

In [ ]:
# Quick GPU check. On the intended runtime this should show two NVIDIA T4 GPUs.
!nvidia-smi


## Install Runtime Stack

Run this cell before importing `torch` or `dgl`. In `auto` mode it installs the pinned CUDA stack only when DGL is missing. Use `force` if an existing DGL/PyTorch install is broken.

The pinned stack mirrors the Colab T4 workflow and avoids common DGL/PyTorch/NumPy incompatibilities.

In [ ]:
import importlib
import subprocess
import sys

INSTALL_MODE = 'auto'  # auto, force, or skip
RESTART_AFTER_INSTALL = False
RUNTIME_STACK_INSTALLED = False

def package_importable(name: str) -> bool:
    try:
        importlib.import_module(name)
        return True
    except Exception as exc:
        print(f'{name} import check failed: {exc}')
        return False

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip'] + list(args)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

need_install = INSTALL_MODE == 'force' or (
    INSTALL_MODE == 'auto' and not package_importable('dgl')
)

if INSTALL_MODE not in {'auto', 'force', 'skip'}:
    raise ValueError("INSTALL_MODE must be 'auto', 'force', or 'skip'.")

if need_install:
    print(sys.version)
    assert sys.version_info[:2] in {(3, 10), (3, 11)}, (
        'Use Python 3.10 or 3.11 for the pinned DGL/PyTorch GPU wheels.'
    )
    pip_install(['uninstall', '-y', 'dgl', 'torch', 'torchdata', 'torchvision', 'torchaudio'])
    pip_install([
        'install', '-q', 'numpy<2', 'scipy', 'sympy', 'PyYAML', 'pydantic',
        'matplotlib', 'tqdm', 'scikit-learn', 'pandas',
    ])
    pip_install([
        'install', '-q', 'torch==2.1.0', 'torchvision==0.16.0',
        'torchaudio==2.1.0', '--index-url', 'https://download.pytorch.org/whl/cu118',
    ])
    pip_install([
        'install', '-q', 'dgl==1.1.3+cu118',
        '-f', 'https://data.dgl.ai/wheels/cu118/repo.html',
    ])
    RUNTIME_STACK_INSTALLED = True
    print('Runtime stack installed.')
elif INSTALL_MODE == 'skip':
    print('Install skipped by INSTALL_MODE=skip.')
else:
    print('DGL is already available; install skipped.')


In [ ]:
# Optional restart after installing. Usually not needed if the install cell ran before ML imports.
if RUNTIME_STACK_INSTALLED and RESTART_AFTER_INSTALL:
    import os
    os.kill(os.getpid(), 9)
elif RUNTIME_STACK_INSTALLED:
    print('Install completed. Continue to the validation cell.')
else:
    print('No runtime restart needed.')


## Fetch Repository And Set Output Paths

This cell clones the GitHub repository into `/kaggle/working` when it is not already present. Use `GIT_REF` only when you want a specific branch, tag, or commit.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

os.environ.setdefault('DGLBACKEND', 'pytorch')

REPO_URL = 'https://github.com/EndritHasani01/fraud-detection-robustness-benchmark.git'
GIT_REF = ''  # optional branch, tag, or commit SHA
PROJECT_DIR = Path('/kaggle/working/fraud-detection-robustness-benchmark')
CLONE_IF_MISSING = True
UPDATE_EXISTING_REPO = False

def looks_like_repo(path: Path) -> bool:
    return (path / 'benchmark' / 'run.py').exists() and (path / 'configs').exists()

if looks_like_repo(Path.cwd()):
    PROJECT_DIR = Path.cwd()
elif looks_like_repo(PROJECT_DIR):
    if UPDATE_EXISTING_REPO:
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(PROJECT_DIR), check=True)
elif CLONE_IF_MISSING:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError('Repository not found. Enable CLONE_IF_MISSING or place the repo under /kaggle/working.')

if GIT_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', GIT_REF], cwd=str(PROJECT_DIR), check=True)
    subprocess.run(['git', 'checkout', 'FETCH_HEAD'], cwd=str(PROJECT_DIR), check=True)

os.chdir(PROJECT_DIR)

RUN_ROOT = Path('/kaggle/working/fraud-benchmark-runs')
OUT_SMOKE = RUN_ROOT / 'gfd_robustness_v3_smoke_kaggle'
OUT_MAIN = RUN_ROOT / 'gfd_robustness_benchmark_v3_kaggle'
OUT_SMOKE.mkdir(parents=True, exist_ok=True)
OUT_MAIN.mkdir(parents=True, exist_ok=True)

os.environ['PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['OUT_SMOKE'] = str(OUT_SMOKE)
os.environ['OUT_MAIN'] = str(OUT_MAIN)

print('Project:', PROJECT_DIR)
print('Smoke output:', OUT_SMOKE)
print('Main output:', OUT_MAIN)


## Research Run Controls

Run the smoke test first. After it passes, set `RUN_MAIN = True` to run the report-facing benchmark. Training is resumable through `results.csv`, so rerunning the same model chunk skips completed keys by default.

The benchmark CLI accepts `--device cuda`, not `cuda:0` or `cuda:1`. This notebook selects a Kaggle GPU for each subprocess by setting `CUDA_VISIBLE_DEVICES` before launching the benchmark.

In [ ]:
CONFIG_SMOKE = 'configs/exp_yelpchi_v3_fast.json'
CONFIG_MAIN = 'configs/exp_yelpchi_v3.json'
CONFIG_FULL = 'configs/exp_yelpchi_v3_full.json'

# Run smoke first. Set RUN_MAIN = True only after smoke succeeds.
RUN_SMOKE = True
RUN_MAIN = False
RUN_SHIFT_PROTOCOL = True

DEVICE = 'cuda'
VISIBLE_GPU_DEFAULT = '0'
MODEL_CHUNKS = ['mlp,sage', 'pmp', 'secgfd']
GPU_ASSIGNMENT = {'mlp,sage': '0', 'pmp': '1', 'secgfd': '0'}

# Keep 0 to use config/default values. Use small values for exploratory Kaggle runs.
MAX_EPOCHS = 0
PATIENCE = 0

# If True, reduce SEC-GFD cost for T4 feasibility. Disclose this if used for final results.
SECGFD_T4_SAFE = False

# Graph generation is deterministic but not resumable like training. Leave false unless rebuilding cached graphs.
FORCE_REBUILD_GRAPHS = False

# Advanced option. The default sequential path is safest because all stages write one results.csv.
RUN_DUAL_GPU_LANES = False

print({
    'RUN_SMOKE': RUN_SMOKE,
    'RUN_MAIN': RUN_MAIN,
    'RUN_SHIFT_PROTOCOL': RUN_SHIFT_PROTOCOL,
    'DEVICE': DEVICE,
    'MODEL_CHUNKS': MODEL_CHUNKS,
    'GPU_ASSIGNMENT': GPU_ASSIGNMENT,
    'MAX_EPOCHS': MAX_EPOCHS,
    'PATIENCE': PATIENCE,
    'SECGFD_T4_SAFE': SECGFD_T4_SAFE,
    'RUN_DUAL_GPU_LANES': RUN_DUAL_GPU_LANES,
})


## Environment Validation

This records package, platform, and GPU state before long runs start. If DGL cannot create a CUDA graph, rerun the install section with the pinned stack.

In [ ]:
import hashlib
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

try:
    import dgl
except Exception as exc:
    raise RuntimeError(
        "DGL failed to import. Run the 'Install Runtime Stack' cell with "
        "INSTALL_MODE='force', then rerun validation."
    ) from exc

print('python:', sys.version)
print('platform:', platform.platform())
print('torch:', torch.__version__)
print('dgl:', dgl.__version__)
print('numpy:', np.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())

try:
    smi = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], text=True).strip()
    print('nvidia-smi devices:')
    print(smi)
except Exception as e:
    print('nvidia-smi query unavailable:', e)

if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(idx)
        cap = torch.cuda.get_device_capability(idx)
        print(f'cuda:{idx}: {name}, capability={cap}')
        try:
            test_graph = dgl.rand_graph(10, 20).to(f'cuda:{idx}')
            print(f'DGL CUDA test graph on cuda:{idx}:', test_graph.num_nodes())
        except Exception as e:
            print(f'DGL CUDA test failed on cuda:{idx}:', e)
elif DEVICE == 'cuda':
    raise RuntimeError('DEVICE is cuda, but torch.cuda.is_available() is false.')


## Config Inspection

This prints the scenario surface, seed counts, model list, and expected run scale before training starts.

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def summarize_config(config_path):
    cfg = load_json(config_path)
    graph_seeds = cfg.get('seeds', {}).get('graph_seeds', cfg.get('seeds', {}).get('training_seeds', []))
    training_seeds = cfg.get('seeds', {}).get('training_seeds', [])
    models = [m.get('model_id') for m in cfg.get('models', [])]
    scenario_rows = []
    for s in cfg.get('scenarios', []):
        scenario_rows.append({
            'scenario_id': s.get('scenario_id'),
            'oracle': bool(s.get('oracle_labels', False)),
            'graph_view_mode': s.get('graph_view_mode', 'canonical'),
            'severity_values': s.get('severity_values', []),
        })
    n_scenario_variants = sum(len(s['severity_values']) * len(graph_seeds) for s in scenario_rows)
    n_clean = len(cfg.get('datasets', [])) * len(cfg.get('data_splits', []))
    n_variants = n_clean + n_scenario_variants * n_clean
    run_count_one_protocol = n_variants * len(training_seeds) * len(models)
    print('config:', config_path)
    print('sha256:', sha256_file(config_path))
    print('experiment:', cfg.get('experiment_name'))
    print('models:', models)
    print('graph seeds:', graph_seeds)
    print('training seeds:', training_seeds)
    print('expected variant rows:', n_variants)
    print('train_on_variant run count:', run_count_one_protocol)
    print('both protocol run count if complete:', run_count_one_protocol * 2)
    display(pd.DataFrame(scenario_rows))
    return cfg

cfg_smoke = summarize_config(CONFIG_SMOKE)
cfg_main = summarize_config(CONFIG_MAIN)


## Live Benchmark Runner

The helper below launches each stage as a subprocess, sets `CUDA_VISIBLE_DEVICES` when requested, streams progress output, and drives a `tqdm` progress bar from the benchmark's `[x/y]` progress lines.

In [ ]:
import re
import shlex

try:
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm

PROGRESS_RE = re.compile(r'\[(\d+)/(\d+)\]')

def clipped(text: str, limit: int = 100) -> str:
    text = str(text).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'

def benchmark_env(visible_gpus=None):
    env = os.environ.copy()
    if visible_gpus not in (None, ''):
        env['CUDA_VISIBLE_DEVICES'] = str(visible_gpus)
    return env

def command_text(cmd) -> str:
    return ' '.join(shlex.quote(str(part)) for part in cmd)

def update_progress_bar(bar, line: str, last_position: int) -> tuple[int, bool]:
    match = PROGRESS_RE.search(line)
    if not match:
        return last_position, False

    done = int(match.group(1))
    total = int(match.group(2))
    if bar.total != total:
        bar.total = total
        bar.unit = 'run'
        bar.refresh()

    increment = max(0, done - last_position)
    if increment:
        bar.update(increment)
    else:
        bar.n = done
        bar.refresh()
    return done, True

def run_benchmark(args, label, cwd=PROJECT_DIR, visible_gpus=None):
    cmd = [sys.executable, '-m', 'benchmark.run'] + [str(arg) for arg in args]
    env = benchmark_env(visible_gpus)
    env_note = f"CUDA_VISIBLE_DEVICES={env.get('CUDA_VISIBLE_DEVICES', '<all>')}"
    start = time.time()

    print()
    print('===', label, '===')
    print('Started:', datetime.now(timezone.utc).isoformat())
    print('Env:', env_note)
    print('Command:', command_text(cmd))
    print()

    bar = tqdm(total=1, desc=label, unit='stage', leave=True)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    saw_run_progress = False
    last_position = 0
    last_line = 'process started'

    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
        stripped = line.strip()
        if stripped:
            last_line = stripped

        last_position, updated = update_progress_bar(bar, line, last_position)
        saw_run_progress = saw_run_progress or updated

        if updated or stripped.startswith(('[preflight]', '[graphs]', '[run]')):
            bar.set_postfix_str(clipped(stripped))

    code = proc.wait()
    elapsed = time.time() - start

    if code != 0:
        bar.set_postfix_str(clipped(f'failed: {last_line}'))
        bar.close()
        raise RuntimeError(f'{label} failed with exit code {code}')

    if saw_run_progress and bar.n < (bar.total or 0):
        bar.update((bar.total or 0) - bar.n)
    elif not saw_run_progress and bar.n < 1:
        bar.update(1 - bar.n)

    bar.set_postfix_str(f'done in {elapsed:.1f}s')
    bar.close()
    print(f'Completed {label} in {elapsed:.1f}s')


## Run Manifest

The manifest captures code, config, environment, and GPU state before execution so the Kaggle run can be audited later.

In [ ]:
def command_output(args, cwd=PROJECT_DIR):
    try:
        return subprocess.check_output(list(args), cwd=str(cwd), text=True, stderr=subprocess.STDOUT).strip()
    except Exception as e:
        return f'unavailable: {e}'

def git_value(args):
    return command_output(['git'] + list(args), cwd=PROJECT_DIR)

def write_manifest(out_dir: Path, config_path: str, label: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        'label': label,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'project_dir': str(PROJECT_DIR),
        'repo_url': REPO_URL,
        'git_ref_requested': GIT_REF,
        'config_path': config_path,
        'config_sha256': sha256_file(config_path),
        'git_commit': git_value(['rev-parse', 'HEAD']),
        'git_status_short': git_value(['status', '--short']),
        'python': sys.version,
        'platform': platform.platform(),
        'torch': torch.__version__,
        'dgl': dgl.__version__,
        'numpy': np.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'cuda_device_count': int(torch.cuda.device_count()),
        'cuda_devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
        'nvidia_smi': command_output(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader']),
        'device_requested': DEVICE,
        'visible_gpu_default': VISIBLE_GPU_DEFAULT,
        'gpu_assignment': GPU_ASSIGNMENT,
        'max_epochs_override': MAX_EPOCHS,
        'patience_override': PATIENCE,
        'secgfd_t4_safe': SECGFD_T4_SAFE,
        'kaggle_kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE', ''),
        'kaggle_url_base': os.environ.get('KAGGLE_URL_BASE', ''),
    }
    path = out_dir / f'kaggle_manifest_{label}.json'
    path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('wrote', path)
    return manifest

if RUN_SMOKE:
    manifest_smoke = write_manifest(OUT_SMOKE, CONFIG_SMOKE, 'smoke')
if RUN_MAIN:
    manifest_main = write_manifest(OUT_MAIN, CONFIG_MAIN, 'main')


## Fast Smoke Test

This validates dataset download, graph caching, baseline training, result writing, and plotting. It intentionally uses the fast v3 config and short training controls.

In [ ]:
if RUN_SMOKE:
    run_benchmark([
        '--config', CONFIG_SMOKE,
        '--stage', 'graphs',
        '--out', OUT_SMOKE,
        '--force',
    ], 'smoke: graph generation')

    run_benchmark([
        '--config', CONFIG_SMOKE,
        '--stage', 'baselines',
        '--out', OUT_SMOKE,
        '--device', DEVICE,
        '--max-epochs', 5,
        '--patience', 2,
        '--force',
    ], 'smoke: baseline training', visible_gpus=VISIBLE_GPU_DEFAULT)

    run_benchmark([
        '--config', CONFIG_SMOKE,
        '--stage', 'plots',
        '--out', OUT_SMOKE,
    ], 'smoke: plots')
else:
    print('Smoke test skipped.')


## Main V3 Graph Cache

Graph generation is deterministic by graph seed. This cell reuses `graph_variants.csv` unless `FORCE_REBUILD_GRAPHS = True`.

In [ ]:
if RUN_MAIN:
    variants_path = OUT_MAIN / 'graph_variants.csv'
    if FORCE_REBUILD_GRAPHS or not variants_path.exists():
        args = [
            '--config', CONFIG_MAIN,
            '--stage', 'graphs',
            '--out', OUT_MAIN,
        ]
        if FORCE_REBUILD_GRAPHS:
            args.append('--force')
        run_benchmark(args, 'main: graph generation')
    else:
        print('Using existing graph cache:', variants_path)
else:
    print('Main benchmark disabled. Set RUN_MAIN = True in the controls cell after smoke passes.')


## Main V3 Training: `train_on_variant`

This protocol retrains each selected model on each graph variant. The default path runs chunks sequentially and pins each subprocess to the GPU listed in `GPU_ASSIGNMENT`.

In [ ]:
def training_args_for_models(models: str):
    args = ['--device', DEVICE]
    if MAX_EPOCHS:
        args += ['--max-epochs', str(MAX_EPOCHS)]
    if PATIENCE:
        args += ['--patience', str(PATIENCE)]
    if SECGFD_T4_SAFE and 'secgfd' in models.split(','):
        args += ['--secgfd-hid-dim', '16', '--secgfd-order-d', '1', '--secgfd-high-order', '1']
    return args

def gpu_for_models(models: str):
    if DEVICE != 'cuda' or not torch.cuda.is_available():
        return None
    if torch.cuda.device_count() <= 1:
        return VISIBLE_GPU_DEFAULT
    return GPU_ASSIGNMENT.get(models, VISIBLE_GPU_DEFAULT)

if RUN_MAIN and not RUN_DUAL_GPU_LANES:
    for models in MODEL_CHUNKS:
        run_benchmark([
            '--config', CONFIG_MAIN,
            '--stage', 'matrix',
            '--models', models,
            '--out', OUT_MAIN,
        ] + training_args_for_models(models), f'main train_on_variant: {models}', visible_gpus=gpu_for_models(models))
elif RUN_MAIN and RUN_DUAL_GPU_LANES:
    print('Sequential train_on_variant skipped because RUN_DUAL_GPU_LANES=True. Use the optional dual-lane section below.')
else:
    print('Main train_on_variant disabled.')


## Main V3 Training: `train_clean_eval_all`

This protocol trains on the clean graph once per split/model/seed and evaluates that trained artifact on every cached variant. Keep it separate from `train_on_variant` in analysis.

In [ ]:
if RUN_MAIN and RUN_SHIFT_PROTOCOL and not RUN_DUAL_GPU_LANES:
    for models in MODEL_CHUNKS:
        run_benchmark([
            '--config', CONFIG_MAIN,
            '--stage', 'matrix',
            '--protocol', 'train_clean_eval_all',
            '--models', models,
            '--out', OUT_MAIN,
        ] + training_args_for_models(models), f'main train_clean_eval_all: {models}', visible_gpus=gpu_for_models(models))
elif RUN_MAIN and RUN_SHIFT_PROTOCOL and RUN_DUAL_GPU_LANES:
    print('Sequential shift protocol skipped because RUN_DUAL_GPU_LANES=True. Use the optional dual-lane section below.')
else:
    print('Main shift protocol disabled.')


## Optional Dual T4 Lane Runner

Use this only if you want both T4s active at the same time. Concurrent processes must not append to the same `results.csv`, so this section writes isolated lane outputs, then merges completed result rows back into `OUT_MAIN/results.csv` after each lane finishes.

Leave `RUN_DUAL_GPU_LANES = False` for the safest report run.

In [ ]:
import csv

RESULT_KEY_COLUMNS = ['dataset_id', 'split_id', 'scenario_id', 'severity', 'graph_seed', 'training_seed', 'model_id', 'protocol']

def result_key(row):
    return tuple(str(row.get(col, '')) for col in RESULT_KEY_COLUMNS)

def prepare_lane_out(lane_name: str) -> Path:
    lane = OUT_MAIN / 'lanes' / lane_name
    lane.mkdir(parents=True, exist_ok=True)
    for rel in ['config.json', 'graph_variants.csv', 'variant_audit.csv']:
        src = OUT_MAIN / rel
        dst = lane / rel
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
    return lane

def merge_lane_results(src_csv: Path, dst_csv: Path):
    if not src_csv.exists():
        print('no lane results to merge:', src_csv)
        return 0
    if not dst_csv.exists():
        shutil.copy2(src_csv, dst_csv)
        with src_csv.open('r', encoding='utf-8') as count_f:
            return max(0, sum(1 for _ in count_f) - 1)
    with dst_csv.open('r', newline='', encoding='utf-8') as f:
        dst_reader = csv.DictReader(f)
        fieldnames = list(dst_reader.fieldnames or [])
        existing = {result_key(row) for row in dst_reader}
    added = 0
    with src_csv.open('r', newline='', encoding='utf-8') as src_f, dst_csv.open('a', newline='', encoding='utf-8') as dst_f:
        src_reader = csv.DictReader(src_f)
        writer = csv.DictWriter(dst_f, fieldnames=fieldnames)
        for row in src_reader:
            key = result_key(row)
            if key in existing:
                continue
            writer.writerow({col: row.get(col, '') for col in fieldnames})
            existing.add(key)
            added += 1
    print(f'merged {added} rows from {src_csv} into {dst_csv}')
    return added

def lane_args(protocol: str, models: str, lane_out: Path):
    args = ['--config', CONFIG_MAIN, '--stage', 'matrix', '--models', models, '--out', lane_out]
    if protocol == 'train_clean_eval_all':
        args += ['--protocol', 'train_clean_eval_all']
    return args + training_args_for_models(models)

def run_lane_group(group):
    procs = []
    for protocol, models, gpu in group:
        safe_models = models.replace(',', '_')
        lane_name = f'{protocol}__{safe_models}'
        lane_out = prepare_lane_out(lane_name)
        cmd = [sys.executable, '-m', 'benchmark.run'] + [str(a) for a in lane_args(protocol, models, lane_out)]
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        log_path = lane_out / f'run_{lane_name}.log'
        log_f = log_path.open('a', encoding='utf-8')
        print('starting lane:', lane_name, 'gpu:', gpu, 'log:', log_path)
        proc = subprocess.Popen(cmd, cwd=str(PROJECT_DIR), env=env, stdout=log_f, stderr=subprocess.STDOUT, text=True)
        procs.append((lane_name, proc, log_f, log_path, lane_out))
    for lane_name, proc, log_f, log_path, lane_out in procs:
        code = proc.wait()
        log_f.close()
        print('finished lane:', lane_name, 'exit:', code, 'log:', log_path)
        if code != 0:
            raise RuntimeError(f'lane {lane_name} failed with exit code {code}; inspect {log_path}')
        merge_lane_results(lane_out / 'results.csv', OUT_MAIN / 'results.csv')

DUAL_LANE_GROUPS = [
    [('train_on_variant', 'mlp,sage', '0'), ('train_on_variant', 'pmp', '1')],
    [('train_on_variant', 'secgfd', '0')],
    [('train_clean_eval_all', 'mlp,sage', '0'), ('train_clean_eval_all', 'pmp', '1')],
    [('train_clean_eval_all', 'secgfd', '0')],
]

if RUN_MAIN and RUN_DUAL_GPU_LANES:
    if not (OUT_MAIN / 'graph_variants.csv').exists():
        raise RuntimeError('Run the main graph cache before dual-lane training.')
    for group in DUAL_LANE_GROUPS:
        if not RUN_SHIFT_PROTOCOL and any(protocol == 'train_clean_eval_all' for protocol, _models, _gpu in group):
            continue
        run_lane_group(group)
else:
    print('Dual-lane runner disabled.')


## Generate Report Artifacts

The plots stage creates summary CSVs, missing/error diagnostics, performance-audit joins, and PNG curves.

In [ ]:
if RUN_MAIN:
    run_benchmark([
        '--config', CONFIG_MAIN,
        '--stage', 'plots',
        '--out', OUT_MAIN,
        '--ci',
    ], 'main: plots and summaries')
else:
    print('Main plot generation disabled.')


## Artifact And Completeness Audit

Run this after smoke or main stages. It checks required files, status counts, error rows, and missing-run diagnostics.

In [ ]:
def audit_outputs(out_dir: Path):
    print('Auditing:', out_dir)
    expected = [
        'config.json',
        'graph_variants.csv',
        'variant_audit.csv',
        'results.csv',
        'plots/summary_curves.csv',
        'plots/performance_drop_max_stress.csv',
        'plots/robustness_scores.csv',
        'plots/audit_curves.csv',
        'plots/performance_audit_join.csv',
        'plots/missing_or_error_runs.csv',
    ]
    file_rows = []
    for rel in expected:
        path = out_dir / rel
        file_rows.append({'artifact': rel, 'exists': path.exists(), 'size_bytes': path.stat().st_size if path.exists() else None})
    display(pd.DataFrame(file_rows))

    results_path = out_dir / 'results.csv'
    if results_path.exists():
        results = pd.read_csv(results_path)
        print('results rows:', len(results))
        if {'protocol', 'model_id', 'status'}.issubset(results.columns):
            display(results.groupby(['protocol', 'model_id', 'status']).size().reset_index(name='n'))
        err = results[results['status'].astype(str).str.lower().eq('error')] if 'status' in results.columns else pd.DataFrame()
        if len(err):
            cols = [c for c in ['protocol', 'model_id', 'scenario_id', 'severity', 'graph_seed', 'training_seed', 'error'] if c in err.columns]
            display(err[cols].head(20))

    missing_path = out_dir / 'plots' / 'missing_or_error_runs.csv'
    if missing_path.exists():
        missing = pd.read_csv(missing_path)
        print('missing/error diagnostic rows:', len(missing))
        display(missing.head(20))

audit_target = OUT_MAIN if RUN_MAIN else OUT_SMOKE
audit_outputs(audit_target)


## Inspect Summary Tables And Curves

Use this for quick sanity checks before writing report conclusions. Do not average across protocols.

In [ ]:
from IPython.display import Image

def inspect_summaries(out_dir: Path):
    plots_dir = out_dir / 'plots'
    summary_path = plots_dir / 'summary_curves.csv'
    audit_path = plots_dir / 'audit_curves.csv'
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        display(summary.head(20))
        group_cols = [c for c in ['protocol', 'model_id', 'metric'] if c in summary.columns]
        if group_cols:
            display(summary.groupby(group_cols).size().reset_index(name='rows'))
    if audit_path.exists():
        audit = pd.read_csv(audit_path)
        display(audit.head(20))
    pngs = sorted(plots_dir.rglob('*.png'))[:8]
    print('showing', len(pngs), 'plot files')
    for path in pngs:
        print(path.relative_to(out_dir))
        display(Image(filename=str(path)))

inspect_summaries(audit_target)


## Retry Error Rows

If a model stage produces `status=error` rows because of a transient Kaggle issue, retry the affected chunk with `--retry-errors`. Keep this cell disabled unless needed.

In [ ]:
RUN_RETRY = False
RETRY_MODELS = 'secgfd'
RETRY_PROTOCOL = 'train_on_variant'  # or train_clean_eval_all

if RUN_RETRY:
    retry_args = [
        '--config', CONFIG_MAIN,
        '--stage', 'matrix',
        '--models', RETRY_MODELS,
        '--out', OUT_MAIN,
        '--retry-errors',
    ] + training_args_for_models(RETRY_MODELS)
    if RETRY_PROTOCOL == 'train_clean_eval_all':
        retry_args += ['--protocol', 'train_clean_eval_all']
    run_benchmark(retry_args, f'retry errors: {RETRY_MODELS} {RETRY_PROTOCOL}', visible_gpus=gpu_for_models(RETRY_MODELS))
else:
    print('Retry disabled.')


## Archive Outputs

Use this after the final plots stage. Kaggle persists `/kaggle/working` as notebook output, but a zip makes downloading or creating a Kaggle dataset easier.

In [ ]:
CREATE_ARCHIVE = False

if CREATE_ARCHIVE:
    import shutil
    archive_base = Path('/kaggle/working') / audit_target.name
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(audit_target))
    print('archive:', archive_path)
else:
    print('Archive disabled.')


## Interpretation Notes

- `results.csv` is the unified performance table.
- `variant_audit.csv` records what each perturbation actually changed.
- `plots/performance_audit_join.csv` joins performance rows with perturbation audit evidence.
- Keep `train_on_variant` and `train_clean_eval_all` separate in all conclusions.
- Disclose oracle scenarios: `heterophily_rewire_oracle`, `camouflage_feature_oracle`, and `camouflage_relation_oracle` use labels during perturbation construction.
- Treat severity as family-specific. Use realized audit metrics when comparing stress strength.
- If `SECGFD_T4_SAFE`, `MAX_EPOCHS`, or `PATIENCE` overrides are used for final results, disclose those runtime controls in the report.